# 17 · 보 종합 설계 — 하중조합부터 상세까지

400 × 700 보(8 m 경간)를 KDS 의 검토 순서대로 설계한다.

```
① 하중조합      KDS 14 20 10 4.2.2
② 내구성·피복   KDS 14 20 40, KDS 14 20 50 4.3
③ 재료·단면     KDS 14 20 10 4.3, KDS 14 20 20 4.1.1
④ 휨 설계       KDS 14 20 20 4.1.2, 4.2.2
⑤ 전단 설계     KDS 14 20 22 4.2, 4.3
⑥ 사용성        KDS 14 20 30 4.2, KDS 14 20 20 4.2.3
⑦ 정착·이음     KDS 14 20 52
```

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [2]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS
from concreteproperties_kds.detailing import (
    bar_area, minimum_bar_spacing, minimum_cover, summarise_detailing,
)
from concreteproperties_kds.durability import check_durability
from concreteproperties_kds.kds import minimum_flexural_moment
from concreteproperties_kds.loads import print_combinations, required_strength
from concreteproperties_kds.serviceability import (
    check_crack_control, check_deflection, minimum_thickness,
)
from concreteproperties_kds.shear import check_shear, required_stirrup_spacing

SPAN, B, H = 8000.0, 400.0, 700.0
FCK, FY, EXPOSURE = 27.0, 400.0, "EC3"
MAIN_BAR, STIRRUP, N_BOT, N_TOP = "D25", "D13", 5, 2

## ① 하중조합 (KDS 14 20 10 4.2.2)

In [3]:
loads = {"D": 22.0, "L": 14.0, "S": 3.0}
print_combinations(loads=loads)

w_u, governing = required_strength(loads=loads)
m_u = w_u * (SPAN / 1000.0) ** 2 / 8 * 1e6
v_u = w_u * (SPAN / 1000.0) / 2 * 1e3

print()
print(f"지배 {governing.name} (식 {governing.equation}) : wu = {w_u:.2f} kN/m")
print(f"Mu = {m_u / 1e6:.2f} kN.m,  Vu = {v_u / 1e3:.2f} kN")

m_sustained = loads["D"] * (SPAN / 1000.0) ** 2 / 8 * 1e6
m_live = loads["L"] * (SPAN / 1000.0) ** 2 / 8 * 1e6

하중조합 (KDS 14 20 10 4.2.2)
     조합        식            U                                                            
------------------------------------------------------------------------------------------------
     U2    4.2-2        50.30  U = 1.2(D+F+T) + 1.6(L + aH*H_v + H_h) + 0.5(L_r or S or R) <= 지배
     U6    4.2-6        50.30  U = 1.2(D+F+T) + 1.6(L + aH*H_v) + 0.8H_h + 0.5(L_r or S or R)
   U3-L    4.2-3        45.20  U = 1.2D + 1.6(L_r or S or R) + 1.0L                      
     U4    4.2-4        41.90  U = 1.2D + 1.3W + 1.0L + 0.5(L_r or S or R)               
   U5-a    4.2-5        41.00  U = 1.2(D+H_v) + 1.0E + 1.0L + 0.2S + 1.0H_h              
   U5-b    4.2-5        41.00  U = 1.2(D+H_v) + 1.0E + 1.0L + 0.2S + 0.5H_h              
   U3-W    4.2-3        31.20  U = 1.2D + 1.6(L_r or S or R) + 0.65W                     
     U1    4.2-1        30.80  U = 1.4(D + F)                                            
   U7-a    4.2-7        19.80  U = 0.9(D + H_v) + 1.3W +

## ② 내구성과 피복두께 (KDS 14 20 40, KDS 14 20 50 4.3)

In [4]:
cover_structural = minimum_cover(condition="옥내_보기둥", fck=FCK)

dur = check_durability(
    exposure_class=EXPOSURE, fck=FCK,
    cover=cover_structural, cover_min=cover_structural,
    water_binder_ratio=0.48,
)
dur.print_results()

d_stirrup, d_main = 12.7, 25.4
cover_to_centre = cover_structural + d_stirrup + d_main / 2
d_eff = H - cover_to_centre

print()
print(f"철근 중심까지 = {cover_to_centre:.1f} mm,  유효깊이 d = {d_eff:.1f} mm")

내구성 검토 (KDS 14 20 40)
노출등급  EC3 (탄산화)
          보통 습도
------------------------------------------------------------------------
설계기준압축강도  fck =     27.0 MPa (표 4.1-3 요구 27.0 이상)  만족
피복두께          cc  =     40.0 mm (KDS 14 20 50 요구 40.0 이상)  만족
물-결합재비       W/B =    0.480   (KCS 14 20 10(1.10) 에서 확인할 것)
------------------------------------------------------------------------
종합                                                만족

철근 중심까지 = 65.4 mm,  유효깊이 d = 634.6 mm


## ③ 재료와 단면

In [5]:
kds = KDS(column_type="tie")
conc = kds.create_concrete_material(compressive_strength=FCK)
steel = kds.create_steel_material(yield_strength=FY)

geom = concrete_rectangular_section(
    d=H, b=B,
    dia_top=15.9, area_top=bar_area("D16"), n_top=N_TOP,
    c_top=cover_to_centre,
    dia_bot=d_main, area_bot=bar_area(MAIN_BAR), n_bot=N_BOT,
    c_bot=cover_to_centre,
    n_circle=16, conc_mat=conc, steel_mat=steel,
)
conc_sec = ConcreteSection(geom)
kds.assign_concrete_section(conc_sec)

a_s = N_BOT * bar_area(MAIN_BAR)
clear_spacing = (
    B - 2 * cover_structural - 2 * d_stirrup - N_BOT * d_main
) / (N_BOT - 1)
s_min = minimum_bar_spacing(bar=MAIN_BAR, member="보", aggregate_size=25)

print(f"단면 {B:.0f} x {H:.0f},  인장 {N_BOT}-{MAIN_BAR} = {a_s:.0f} mm^2")
print(f"{conc.name},  Ec = {conc.elastic_modulus:,.0f} MPa")
print(f"철근 순간격 {clear_spacing:.1f} mm >= 최소 {s_min:.1f} mm"
      f"  {'만족' if clear_spacing >= s_min else '불만족'}")

conc_sec.plot_section()

단면 400 x 700,  인장 5-D25 = 2534 mm^2
fck 27 MPa 콘크리트 (KDS 14 20),  Ec = 26,702 MPa
철근 순간격 41.9 mm >= 최소 33.3 mm  만족


/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53080 (\N{HANGUL SYLLABLE KON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 47532 (\N{HANGUL SYLLABLE RI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53944 (\N{HANGUL SYLLABL

<Axes: title={'center': 'Reinforced Concrete Section'}>

## ④ 휨 설계 (KDS 14 20 20)

In [6]:
f_res, u_res, phi = kds.ultimate_bending_capacity(theta=0, n_design=0)
eps_t = kds.net_tensile_strain(theta=0, d_n=u_res.d_n)

print(f"중립축 깊이   c      = {u_res.d_n:8.2f} mm")
print(f"순인장변형률  et     = {eps_t:8.5f}  "
      f"({kds.section_classification(eps_t=eps_t)})")
print(f"강도감소계수  phi    = {phi:8.3f}")
print(f"설계 휨강도 phi*Mn   = {f_res.m_x / 1e6:8.2f} kN.m")
print(f"소요 휨모멘트  Mu    = {m_u / 1e6:8.2f} kN.m")
print(f"소요/강도            = {m_u / f_res.m_x:8.3f}"
      f"  {'만족' if f_res.m_x >= m_u else '불만족'}")
print()

_, eps_min, ok_duct = kds.check_flexural_ductility()
phi_m_n, m_cr, _, ok_min = kds.check_minimum_flexural_reinforcement()
print(f"최소허용변형률 et,min = {eps_min:8.5f}  "
      f"{'만족' if ok_duct else '불만족'}   (4.1.2(5))")
print(f"최소 철근량  1.2*Mcr  = "
      f"{minimum_flexural_moment(m_cr=m_cr) / 1e6:8.2f} kN.m  "
      f"{'만족' if ok_min else '불만족'}   (4.2.2)")

중립축 깊이   c      =   124.56 mm
순인장변형률  et     =  0.01318  (인장지배단면)
강도감소계수  phi    =    0.850
설계 휨강도 phi*Mn   =   490.81 kN.m
소요 휨모멘트  Mu    =   402.40 kN.m
소요/강도            =    0.820  만족



최소허용변형률 et,min =  0.00400  만족   (4.1.2(5))
최소 철근량  1.2*Mcr  =   149.01 kN.m  만족   (4.2.2)


## ⑤ 전단 설계 (KDS 14 20 22)

In [7]:
a_v = 2 * bar_area(STIRRUP)
s_req = required_stirrup_spacing(
    v_u=v_u, fck=FCK, b_w=B, d=d_eff, a_v=a_v, fyt=FY
)
s_use = min(25.0 * int(s_req / 25.0), 250.0)

print(f"스터럽 {STIRRUP} 2가닥, 필요 {s_req:.1f} mm -> 배치 {s_use:.0f} mm")
print()

shear = check_shear(
    v_u=v_u, fck=FCK, b_w=B, d=d_eff, a_v=a_v, s=s_use, fyt=FY
)
shear.print_results()

스터럽 D13 2가닥, 필요 317.3 mm -> 배치 250 mm

전단 검토 (KDS 14 20 22)
계수 전단력          Vu      =     201.20 kN
콘크리트 전단강도    Vc      =     219.83 kN
                 phi*Vc      =     164.87 kN
전단철근 전단강도    Vs      =     257.29 kN
                     Vs,max  =     879.33 kN
공칭 전단강도        Vn      =     477.12 kN
설계 전단강도    phi*Vn      =     357.84 kN
------------------------------------------------------------------
전단철근 필요                = 예
배치 전단철근량      Av      =     253.40 mm^2
최소 전단철근량      Av,min  =      87.50 mm^2
배치 간격            s       =     250.00 mm
최대 간격            s,max   =     317.30 mm
------------------------------------------------------------------
강도       phi*Vn >= Vu      : 만족
최소철근   Av >= Av,min      : 만족
간격       s <= s,max        : 만족
철근한계   Vs <= Vs,max      : 만족
단면크기                     : 만족
종합                         : 만족


## ⑥ 사용성 (KDS 14 20 30)

In [8]:
h_min = minimum_thickness(span=SPAN, member="보", support="단순지지", fy=FY)
print(f"최소 두께 l/16 = {h_min:.1f} mm, h = {H:.1f} mm"
      f"  ->  {'생략 가능' if h_min <= H else '처짐 계산 필요'}")
print()

gross = kds.get_transformed_gross_properties(
    elastic_modulus=conc.elastic_modulus
)
cracked = kds.calculate_cracked_properties(theta=0)
cracked.calculate_transformed_properties(
    elastic_modulus=conc.elastic_modulus
)

defl = check_deflection(
    span=SPAN, m_sustained=m_sustained, m_live=m_live,
    m_cr=cracked.m_cr, i_g=gross.ixx_c, i_cr=cracked.ixx_c_cr,
    e_c=conc.elastic_modulus,
    rho_prime=N_TOP * bar_area("D16") / (B * d_eff),
)
defl.print_results()

bar_spacing = (
    B - 2 * cover_structural - 2 * d_stirrup - d_main
) / (N_BOT - 1)
fs, s_max, ok_crack = check_crack_control(
    bar_spacing=bar_spacing, fy=FY, c_c=cover_structural + d_stirrup
)
print()
print(f"균열 제어  s = {bar_spacing:.1f} <= s,max = {s_max:.1f} mm"
      f"  {'만족' if ok_crack else '불만족'}   (14 20 20 4.2.3(4))")

최소 두께 l/16 = 500.0 mm, h = 700.0 mm  ->  생략 가능

처짐 검토 (KDS 14 20 30 4.2)
총단면 2차모멘트     Ig    =   12,799,725,708 mm^4
균열단면 2차모멘트   Icr   =    4,485,580,430 mm^4
유효단면2차모멘트    Ie    =    5,151,995,146 mm^4
                     Ie/Ig =            0.403
------------------------------------------------------------------
지속하중 즉시처짐          =            8.529 mm
활하중   즉시처짐          =            5.428 mm
장기 추가처짐              =           15.821 mm
전체 처짐 (참고)           =           29.777 mm
------------------------------------------------------------------
허용처짐 조건 : 바닥_비구조재없음
비교 대상     : 활하중에 의한 즉시처짐
검토 처짐                  =            5.428 mm
허용 처짐                  =           22.222 mm
판정                       =               만족

균열 제어  s = 67.3 <= s,max = 262.0 mm  만족   (14 20 20 4.2.3(4))


## ⑦ 정착·이음 (KDS 14 20 52)

In [9]:
summarise_detailing(bar=MAIN_BAR, fy=FY, fck=FCK).print_results()

정착·이음 길이 - D25 (KDS 14 20 52)
공칭 지름              db   =     25.40 mm
인장 정착길이          ld   =    1173.2 mm
압축 정착길이          ldc  =     488.8 mm
표준갈고리 정착길이    ldh  =     469.3 mm
인장 겹침이음 (A급)         =    1173.2 mm
인장 겹침이음 (B급)         =    1525.1 mm
압축 겹침이음               =     731.5 mm


## 설계 요약

In [10]:
items = [
    ("휨강도 (14 20 20 4.1)", f_res.m_x >= m_u),
    ("연성 (14 20 20 4.1.2(5))", ok_duct),
    ("최소 철근량 (14 20 20 4.2.2)", ok_min),
    ("전단강도 (14 20 22)", shear.ok),
    ("처짐 (14 20 30 4.2)", defl.ok),
    ("균열 제어 (14 20 20 4.2.3)", ok_crack),
    ("내구성 (14 20 40)", dur.ok),
    ("철근 순간격 (14 20 50 4.2)", clear_spacing >= s_min),
]

for name, ok in items:
    print(f"  {name:<32} {'만족' if ok else '불만족'}")
print()
print(f"  {'종합':<32} "
      f"{'만족' if all(ok for _, ok in items) else '불만족'}")

  휨강도 (14 20 20 4.1)               만족
  연성 (14 20 20 4.1.2(5))           만족
  최소 철근량 (14 20 20 4.2.2)          만족
  전단강도 (14 20 22)                  만족
  처짐 (14 20 30 4.2)                만족
  균열 제어 (14 20 20 4.2.3)           만족
  내구성 (14 20 40)                   만족
  철근 순간격 (14 20 50 4.2)            만족

  종합                               만족
